In [3]:
import pandas as pd
import os
from pathlib import Path

def merge_csv_from_subfolders(root_folder="../artifacts", output_path="../artifacts/runs.csv", pattern="*.csv", recursive=True):
    """
    Объединяет CSV файлы из всех вложенных папок в один DataFrame.
    
    Parameters:
    -----------
    root_folder : str
        Путь к корневой папке, содержащей вложенные папки с CSV файлами
    output_path : str
        Путь для сохранения объединённого CSV файла
    pattern : str
        Шаблон имени файла (по умолчанию "*.csv")
    recursive : bool
        Искать рекурсивно во всех подпапках (по умолчанию True)
    
    Returns:
    --------
    pd.DataFrame
        Объединённый DataFrame
    """
    
    # Список для хранения всех DataFrames
    dfs = []
    
    # Счётчик найденных файлов
    file_count = 0
    
    # Поиск файлов
    if recursive:
        # Рекурсивный поиск во всех подпапках
        for filepath in Path(root_folder).rglob(pattern):
            try:
                # Читаем CSV
                df = pd.read_csv(filepath)
                
                dfs.append(df)
                file_count += 1
                print(f"Загружен: {filepath}")
                
            except Exception as e:
                print(f"Ошибка при загрузке {filepath}: {e}")
    else:
        # Поиск только в корневой папке
        for filepath in Path(root_folder).glob(pattern):
            try:
                df = pd.read_csv(filepath)
                df['source_file'] = filepath.name
                df['source_path'] = str(filepath.parent)
                dfs.append(df)
                file_count += 1
                print(f"✓ Загружен: {filepath}")
            except Exception as e:
                print(f"✗ Ошибка при загрузке {filepath}: {e}")
    
    if not dfs:
        print(f"Не найдено ни одного файла с шаблоном '{pattern}' в {root_folder}")
        return None
    
    # Объединяем все DataFrames
    merged_df = pd.concat(dfs, ignore_index=True)
    
    print(f"\n--- Результаты ---")
    print(f"Найдено файлов: {file_count}")
    print(f"Всего строк в объединённом файле: {len(merged_df)}")
    print(f"Колонки: {list(merged_df.columns)}")
    
    # Создаём директорию для сохранения, если её нет
    
    # Сохраняем результат
    merged_df.to_csv(output_path, index=False)
    print(f"Объединённый файл сохранён: {output_path}")
    return merged_df


# Пример использования
if __name__ == "__main__":
    # Вариант 1: базовая настройка    
    df_merged = merge_csv_from_subfolders()
    

Загружен: ..\artifacts\runs.csv
Загружен: ..\artifacts\Baselines\runsBase.csv
Загружен: ..\artifacts\ExperimentBalanced\runsBalanced.csv
Загружен: ..\artifacts\ExperimentBoost\runsExpB.csv
Загружен: ..\artifacts\ExperimentMLP\runsMLP.csv
Загружен: ..\artifacts\Experiments\runsExp.csv

--- Результаты ---
Найдено файлов: 6
Всего строк в объединённом файле: 32
Колонки: ['accuracy', 'f1', 'Precision', 'Recall', 'roc_auc', 'model', 'precision', 'recall']
Объединённый файл сохранён: ../artifacts/runs.csv
